# Build a BIDS eye-tracking dataset

This notebook converts the current DeepGazeIR EyeLink data into a BIDS 1.11.1 dataset.

It uses the existing project classes:

- `Loader`
- `Parser`
- `Extractor`
- `Profiler`
- `Cleaner`

The current parser extracts trial, fixation, and blink events but does not retain continuous gaze samples. A small ASC sample reader is therefore included below to create the mandatory BIDS `*_physio.tsv.gz` files.

Raw device data are stored in the BIDS dataset. Cleaned fixation tables are stored separately under `derivatives/deepgazeir-processing/`.


## Output structure

```text
bids_eyetracking/
├── dataset_description.json
├── participants.tsv
├── participants.json
├── README
├── stimuli/
│   ├── visible/
│   └── infrared/
├── sub-001/
│   └── beh/
│       ├── sub-001_task-freeview_acq-visible_run-01_events.tsv
│       ├── sub-001_task-freeview_acq-visible_run-01_events.json
│       ├── sub-001_task-freeview_acq-visible_run-01_recording-eye1_physio.tsv.gz
│       ├── sub-001_task-freeview_acq-visible_run-01_recording-eye1_physio.json
│       ├── sub-001_task-freeview_acq-visible_run-01_recording-eye1_physioevents.tsv.gz
│       └── sub-001_task-freeview_acq-visible_run-01_recording-eye1_physioevents.json
└── derivatives/
    └── deepgazeir-processing/
        ├── dataset_description.json
        └── sub-001/
            └── beh/
                └── sub-001_task-freeview_acq-visible_run-01_desc-clean_fixations.tsv
```

Each ASC file is treated as one BIDS run. If a participant has several ASC files, runs are numbered automatically.


In [5]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parents[0]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))
    
print(PROJECT_ROOT)

/Users/noe-edouard/Desktop/ETUDES/INP/Stages/3A/Code


In [9]:
from __future__ import annotations

from pathlib import Path
import json
import re
import shutil
import warnings

import numpy as np
import pandas as pd

from helpers.loader import Loader
from processing.parser import Parser
from processing.extractor import Extractor
from processing.profiler import Profiler
from processing.cleaner import Cleaner

EYETRACKING_DIR = PROJECT_ROOT / "eyetracking"
DATA_DIR = PROJECT_ROOT / "data" / "2026" 


EDF_DIR = DATA_DIR / "global" / "edf"
ASC_DIR = DATA_DIR / "global" / "asc"
IMAGES_DIR = DATA_DIR / "global" / "images"
TABLES_DIR = DATA_DIR / "global" / "tables"
BIDS_DIR = DATA_DIR / "bids"

METADATA_PATH = DATA_DIR / "global" / "metadata" / "metadata.json"
PARTICIPANTS_PATH = DATA_DIR / "global" / "tables" / "participants.csv"
ASC_ROOTS = {"infrared": ASC_DIR / "infrared", "visible": ASC_DIR / "visible"}
STIMULI_ROOTS = {"infrared": IMAGES_DIR / "infrared", "visible": IMAGES_DIR / "visible"}

TASK_LABEL = "freeview"
BIDS_VERSION = "1.11.1"
DATASET_NAME = "DeepGazeIR Eye-Tracking Free-Viewing Dataset"
AUTHORS = ["Noé Edouard", "Jocelyn Chanussot", "Nathalie Guyader", "Ronald Phlypo", "Loïc Arbez"]

SCREEN_DISTANCE_M = 0.70
SCREEN_WIDTH_M = 0.535
SCREEN_REFRESH_RATE_HZ = 60.0
SCREEN_ORIGIN = ["top", "left"]

MIN_FIXATION_DURATION_MS = 80
MAX_BLINKS_PER_TRIAL = 3
CENTER_TOLERANCE_DEG = 2.0
MIN_FIXATIONS_PER_TRIAL = 3

COPY_STIMULI = True
FILL_RECORDING_GAPS = False
OVERWRITE_BIDS_DIR = True


In [10]:
_NUMBER = re.compile(r"^[+-]?\d+(?:\.\d+)?$")


def sanitize_bids_label(value: object) -> str:
    label = re.sub(r"[^A-Za-z0-9]+", "", str(value))
    if not label:
        raise ValueError(f"Invalid BIDS label: {value!r}")
    return label


def subject_label(value: object) -> str:
    match = re.search(r"\d+", str(value))
    if match:
        return f"{int(match.group()):03d}"
    return sanitize_bids_label(value)


def read_table_auto(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, sep=None, engine="python")
    return df.dropna(axis=1, how="all")


def write_json(data: dict, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as stream:
        json.dump(data, stream, indent=2, ensure_ascii=False)
        stream.write("\n")


def write_tsv(df: pd.DataFrame, path: Path, header: bool = True) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(
        path,
        sep="\t",
        index=False,
        header=header,
        na_rep="n/a",
        lineterminator="\n",
    )


def write_tsv_gz(df: pd.DataFrame, path: Path, header: bool = False) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(
        path,
        sep="\t",
        index=False,
        header=header,
        na_rep="n/a",
        compression="gzip",
        lineterminator="\n",
    )


def discover_recordings() -> pd.DataFrame:
    rows = []

    for modality, root in ASC_ROOTS.items():
        if root is None or not root.exists():
            continue

        files = sorted({*root.rglob("*.asc"), *root.rglob("*.ASC")})
        for filepath in files:
            participant = subject_label(Extractor._get_participant_id(filepath.name))
            rows.append({
                "participant_id": participant,
                "modality": modality,
                "asc_path": filepath,
            })

    recordings = pd.DataFrame(rows)

    if recordings.empty:
        searched = "\n".join(
            f"- {modality}: {root}"
            for modality, root in ASC_ROOTS.items()
        )
        raise FileNotFoundError(
            "No ASC file was found recursively.\n"
            "Edit ASC_ROOTS in the configuration cell.\n"
            f"Searched locations:\n{searched}"
        )

    recordings = recordings.sort_values(
        ["participant_id", "modality", "asc_path"]
    ).reset_index(drop=True)

    recordings["run"] = (
        recordings.groupby(["participant_id", "modality"])
        .cumcount()
        .add(1)
    )

    return recordings


def _to_float(value: str) -> float:
    if value in {".", "..", "...", "MISSING_DATA", "NaN", "nan"}:
        return np.nan
    try:
        return float(value)
    except ValueError:
        return np.nan


def _sample_configuration(lines: list[str]) -> tuple[list[str], float | None]:
    eyes: list[str] = []
    sampling_frequency = None

    for line in lines:
        tokens = line.strip().upper().split()
        if not tokens or tokens[0] != "SAMPLES":
            continue

        eyes = [eye.lower() for eye in ("LEFT", "RIGHT") if eye in tokens]

        if "RATE" in tokens:
            index = tokens.index("RATE")
            if index + 1 < len(tokens):
                try:
                    sampling_frequency = float(tokens[index + 1])
                except ValueError:
                    pass
        break

    return eyes, sampling_frequency


def _regularize_samples(
    df: pd.DataFrame,
    sampling_frequency: float,
) -> pd.DataFrame:
    if df.empty or not FILL_RECORDING_GAPS:
        return df

    step_ms = 1000.0 / float(sampling_frequency)
    start = float(df["timestamp"].min())
    stop = float(df["timestamp"].max())

    n_rows = int(round((stop - start) / step_ms)) + 1
    if n_rows > 10_000_000:
        raise RuntimeError(
            f"Refusing to create {n_rows:,} rows while filling recording gaps."
        )

    expected = start + np.arange(n_rows, dtype=np.float64) * step_ms
    decimals = 6
    base = pd.DataFrame({"timestamp": np.round(expected, decimals)})

    observed = df.copy()
    observed["timestamp"] = observed["timestamp"].round(decimals)
    observed = observed.drop_duplicates("timestamp", keep="first")

    return base.merge(observed, on="timestamp", how="left")


def read_eyelink_samples(
    asc_path: Path,
    fallback_eye: str | None = None,
) -> tuple[dict[str, pd.DataFrame], float]:
    """Read the standard timestamp/x/y/pupil columns from EyeLink ASC sample rows."""
    lines = asc_path.read_text(encoding="utf-8", errors="ignore").splitlines()
    eyes, sampling_frequency = _sample_configuration(lines)

    if not eyes:
        fallback = str(fallback_eye or "").upper()
        if fallback.startswith("L"):
            eyes = ["left"]
        elif fallback.startswith("R"):
            eyes = ["right"]

    rows: dict[str, list[dict]] = {"left": [], "right": []}

    for line in lines:
        parts = line.strip().split()
        if len(parts) < 4 or not _NUMBER.fullmatch(parts[0]):
            continue

        timestamp = _to_float(parts[0])
        if not np.isfinite(timestamp):
            continue

        if set(eyes) == {"left", "right"} or (not eyes and len(parts) >= 7):
            if len(parts) < 7:
                continue
            rows["left"].append({
                "timestamp": timestamp,
                "x_coordinate": _to_float(parts[1]),
                "y_coordinate": _to_float(parts[2]),
                "pupil_size": _to_float(parts[3]),
            })
            rows["right"].append({
                "timestamp": timestamp,
                "x_coordinate": _to_float(parts[4]),
                "y_coordinate": _to_float(parts[5]),
                "pupil_size": _to_float(parts[6]),
            })
        else:
            eye = eyes[0] if eyes else "right"
            rows[eye].append({
                "timestamp": timestamp,
                "x_coordinate": _to_float(parts[1]),
                "y_coordinate": _to_float(parts[2]),
                "pupil_size": _to_float(parts[3]),
            })

    streams = {
        eye: pd.DataFrame(values)
        for eye, values in rows.items()
        if values
    }

    if not streams:
        raise ValueError(
            f"No continuous EyeLink sample row was found in {asc_path.name}. "
            "Re-export the EDF with samples enabled."
        )

    if sampling_frequency is None:
        first_stream = next(iter(streams.values()))
        differences = first_stream["timestamp"].sort_values().diff()
        differences = differences[differences > 0]
        if differences.empty:
            raise ValueError(f"Unable to infer sampling frequency for {asc_path.name}")
        sampling_frequency = 1000.0 / float(differences.median())

    streams = {
        eye: _regularize_samples(
            df.sort_values("timestamp").reset_index(drop=True),
            sampling_frequency,
        )
        for eye, df in streams.items()
    }

    return streams, float(sampling_frequency)


def eye_mask(series: pd.Series, eye: str) -> pd.Series:
    values = series.astype(str).str.upper().str.strip()
    if eye == "left":
        return values.str.startswith("L")
    if eye == "right":
        return values.str.startswith("R")
    return pd.Series(False, index=series.index)


def build_physioevents(
    fixations: pd.DataFrame,
    blinks: pd.DataFrame,
    eye: str,
) -> pd.DataFrame:
    columns = [
        "onset",
        "duration",
        "trial_type",
        "blink",
        "x_coordinate",
        "y_coordinate",
        "image_name",
        "image_modality",
        "trial_key",
        "event_id",
    ]
    frames = []

    if not fixations.empty:
        selected = fixations.loc[eye_mask(fixations["eye"], eye)].copy()
        if not selected.empty:
            selected = selected.assign(
                onset=selected["start"],
                trial_type="fixation",
                blink=0,
                x_coordinate=selected["x"],
                y_coordinate=selected["y"],
                event_id=selected["fixation_id"],
            )
            frames.append(selected[[
                "onset", "duration", "trial_type", "blink",
                "x_coordinate", "y_coordinate", "image_name",
                "image_modality", "trial_key", "event_id",
            ]])

    if not blinks.empty:
        selected = blinks.loc[eye_mask(blinks["eye"], eye)].copy()
        if not selected.empty:
            selected = selected.assign(
                onset=selected["start"],
                trial_type="blink",
                blink=1,
                x_coordinate=np.nan,
                y_coordinate=np.nan,
                event_id=selected["blink_id"],
            )
            frames.append(selected[[
                "onset", "duration", "trial_type", "blink",
                "x_coordinate", "y_coordinate", "image_name",
                "image_modality", "trial_key", "event_id",
            ]])

    if not frames:
        return pd.DataFrame(columns=columns)

    return (
        pd.concat(frames, ignore_index=True)
        .sort_values(["onset", "trial_type", "event_id"])
        .reset_index(drop=True)
    )


def copy_referenced_stimuli(events: pd.DataFrame) -> None:
    if not COPY_STIMULI:
        return

    for _, row in events[["modality", "image_name"]].drop_duplicates().iterrows():
        modality = str(row["modality"])
        image_name = str(row["image_name"])
        source_root = STIMULI_ROOTS.get(modality)

        if source_root is None:
            warnings.warn(f"No stimulus root configured for {modality}.")
            continue

        source = source_root / image_name
        if not source.exists():
            matches = list(source_root.rglob(image_name))
            source = matches[0] if matches else source

        if not source.exists():
            warnings.warn(f"Stimulus not found: {modality}/{image_name}")
            continue

        destination = BIDS_DIR / "stimuli" / modality / image_name
        destination.parent.mkdir(parents=True, exist_ok=True)
        if not destination.exists():
            shutil.copy2(source, destination)


recordings = discover_recordings()
recordings

,participant_id,modality,asc_path,run
0,001,infrared,/Users/noe-edouard/Desktop/ETUDES/INP/Stages/3...,1
1,002,infrared,/Users/noe-edouard/Desktop/ETUDES/INP/Stages/3...,1
2,003,visible,/Users/noe-edouard/Desktop/ETUDES/INP/Stages/3...,1
3,004,infrared,/Users/noe-edouard/Desktop/ETUDES/INP/Stages/3...,1
4,005,infrared,/Users/noe-edouard/Desktop/ETUDES/INP/Stages/3...,1
...,...,...,...,...
63,068,infrared,/Users/noe-edouard/Desktop/ETUDES/INP/Stages/3...,1
64,069,visible,/Users/noe-edouard/Desktop/ETUDES/INP/Stages/3...,1
65,070,infrared,/Users/noe-edouard/Desktop/ETUDES/INP/Stages/3...,1
66,071,infrared,/Users/noe-edouard/Desktop/ETUDES/INP/Stages/3...,1


In [12]:
loader = Loader(METADATA_PATH)
parser = Parser(loader.metadata)
extractor = Extractor(parser)
cleaner = Cleaner(
    min_fix_duration=MIN_FIXATION_DURATION_MS,
    keep_only_inside_screen=True,
    remove_initial_fixation=True,
    initial_fixation_id=0,
)

if OVERWRITE_BIDS_DIR and BIDS_DIR.exists():
    for item in BIDS_DIR.iterdir():
        if item.is_dir():
            shutil.rmtree(item)
        else:
            item.unlink()

BIDS_DIR.mkdir(parents=True, exist_ok=True)

write_json(
    {
        "Name": DATASET_NAME,
        "BIDSVersion": BIDS_VERSION,
        "DatasetType": "raw",
        "License": "n/a",
        "Authors": AUTHORS,
    },
    BIDS_DIR / "dataset_description.json",
)

derivative_root = BIDS_DIR / "derivatives" / "deepgazeir-processing"
write_json(
    {
        "Name": "DeepGazeIR eye-tracking processing derivatives",
        "BIDSVersion": BIDS_VERSION,
        "DatasetType": "derivative",
        "GeneratedBy": [
            {
                "Name": "DeepGazeIR processing pipeline",
                "Description": "Parser, Extractor, Profiler and Cleaner",
            }
        ],
    },
    derivative_root / "dataset_description.json",
)

source_participants = pd.DataFrame()
if PARTICIPANTS_PATH is not None:
    source_participants = read_table_auto(PARTICIPANTS_PATH)
    if "id" in source_participants.columns:
        source_participants["participant_id"] = source_participants["id"].map(subject_label)

participant_rows = []
build_rows = []

for recording in recordings.itertuples(index=False):
    participant = str(recording.participant_id)
    modality = str(recording.modality)
    run = int(recording.run)
    asc_path = Path(recording.asc_path)

    trials = parser.parse_file(
        filepath=asc_path,
        participant_id=participant,
        modality=modality,
    )
    if not trials:
        raise ValueError(f"No valid trial found in {asc_path}")

    raw_summary = extractor.extract_summary(trials)
    raw_fixations = extractor.extract_fixations(trials)
    raw_blinks = extractor.extract_blinks(trials)

    screen_width_px = int(raw_summary["screen_width"].iloc[0])
    screen_height_px = int(raw_summary["screen_height"].iloc[0])
    screen_height_m = SCREEN_WIDTH_M * screen_height_px / screen_width_px

    profiler = Profiler(
        min_fix_duration=MIN_FIXATION_DURATION_MS,
        max_blinks_valid=MAX_BLINKS_PER_TRIAL,
        center_tolerance_deg=CENTER_TOLERANCE_DEG,
        n_min_fixation_per_trial=MIN_FIXATIONS_PER_TRIAL,
        viewing_distance_cm=SCREEN_DISTANCE_M * 100.0,
        screen_width_cm=SCREEN_WIDTH_M * 100.0,
        screen_height_cm=screen_height_m * 100.0,
    )

    trial_quality, participant_summary, invalid_trials = profiler.run(
        summary_df=raw_summary,
        fixations_df=raw_fixations,
        blinks_df=raw_blinks,
        verbose=False,
        save=False,
    )

    clean_fixations, clean_summary = cleaner.run(
        raw_fixations_df=raw_fixations,
        trial_summary_df=trial_quality,
        verbose=False,
        save=False,
    )

    fallback_eye = None
    if not raw_fixations.empty and raw_fixations["eye"].nunique() == 1:
        fallback_eye = str(raw_fixations["eye"].iloc[0])

    streams, sampling_frequency = read_eyelink_samples(
        asc_path,
        fallback_eye=fallback_eye,
    )

    first_timestamp = min(float(df["timestamp"].min()) for df in streams.values())

    prefix = (
        f"sub-{participant}_task-{TASK_LABEL}"
        f"_acq-{sanitize_bids_label(modality)}_run-{run:02d}"
    )
    beh_dir = BIDS_DIR / f"sub-{participant}" / "beh"
    beh_dir.mkdir(parents=True, exist_ok=True)

    events = raw_summary.copy()
    events["onset"] = (events["image_onset"] - first_timestamp) / 1000.0
    events["duration"] = events["image_duration"] / 1000.0
    events["trial_type"] = "image"
    events["stim_file"] = (
        events["image_modality"].astype(str)
        + "/"
        + events["image_name"].astype(str)
    )
    events = events.rename(columns={"image_modality": "modality"})
    events = events[[
        "onset",
        "duration",
        "trial_type",
        "stim_file",
        "modality",
        "image_id",
        "image_stem",
        "image_name",
        "trial_index",
    ]]

    write_tsv(events, beh_dir / f"{prefix}_events.tsv", header=True)
    write_json(
        {
            "TaskName": "Free viewing of visible and infrared images",
            "StimulusPresentation": {
                "ScreenDistance": SCREEN_DISTANCE_M,
                "ScreenOrigin": SCREEN_ORIGIN,
                "ScreenRefreshRate": SCREEN_REFRESH_RATE_HZ,
                "ScreenResolution": [screen_width_px, screen_height_px],
                "ScreenSize": [SCREEN_WIDTH_M, screen_height_m],
            },
            "onset": {
                "Description": "Image onset relative to the first eye-tracking sample.",
                "Units": "s",
            },
            "duration": {
                "Description": "Image presentation duration.",
                "Units": "s",
            },
            "trial_type": {"Description": "Experimental event type."},
            "stim_file": {
                "Description": "Path relative to the BIDS stimuli directory."
            },
            "modality": {
                "Description": "Image modality.",
                "Levels": {
                    "visible": "Visible RGB stimulus.",
                    "infrared": "Long-wave infrared stimulus.",
                },
            },
            "image_id": {"Description": "Project image identifier."},
            "image_stem": {"Description": "Shared scene identifier."},
            "image_name": {"Description": "Canonical image filename."},
            "trial_index": {"Description": "Trial order in the ASC recording."},
        },
        beh_dir / f"{prefix}_events.json",
    )

    copy_referenced_stimuli(events)

    available_eyes = [eye for eye in ("left", "right") if eye in streams]
    for eye_index, eye in enumerate(available_eyes, start=1):
        recording_label = f"eye{eye_index}"
        physio_stem = f"{prefix}_recording-{recording_label}_physio"

        physio = streams[eye][[
            "timestamp",
            "x_coordinate",
            "y_coordinate",
            "pupil_size",
        ]]

        write_tsv_gz(
            physio,
            beh_dir / f"{physio_stem}.tsv.gz",
            header=False,
        )
        write_json(
            {
                "SamplingFrequency": sampling_frequency,
                "StartTime": 0.0,
                "Columns": [
                    "timestamp",
                    "x_coordinate",
                    "y_coordinate",
                    "pupil_size",
                ],
                "Manufacturer": "SR Research",
                "PhysioType": "eyetrack",
                "RecordedEye": eye,
                "SampleCoordinateSystem": "gaze-on-screen",
                "EyeTrackingMethod": "P-CR",
                "CalibrationType": "HV9",
                "timestamp": {
                    "Description": "EyeLink device timestamp.",
                    "Units": "ms",
                    "Origin": "EyeLink tracker clock",
                },
                "x_coordinate": {
                    "LongName": "Gaze position x-coordinate",
                    "Description": "Horizontal gaze position on the screen.",
                    "Units": "pixel",
                },
                "y_coordinate": {
                    "LongName": "Gaze position y-coordinate",
                    "Description": "Vertical gaze position on the screen.",
                    "Units": "pixel",
                },
                "pupil_size": {
                    "Description": "Pupil area reported by EyeLink.",
                    "Units": "arbitrary",
                },
            },
            beh_dir / f"{physio_stem}.json",
        )

        physioevents = build_physioevents(
            raw_fixations,
            raw_blinks,
            eye=eye,
        )
        if not physioevents.empty:
            events_stem = f"{prefix}_recording-{recording_label}_physioevents"
            write_tsv_gz(
                physioevents,
                beh_dir / f"{events_stem}.tsv.gz",
                header=False,
            )
            write_json(
                {
                    "Columns": list(physioevents.columns),
                    "Description": "EyeLink fixation and blink events.",
                    "OnsetSource": "timestamp",
                    "onset": {
                        "Description": "EyeLink device timestamp at event onset.",
                        "Units": "ms",
                    },
                    "duration": {
                        "Description": "Event duration.",
                        "Units": "ms",
                    },
                    "trial_type": {
                        "Description": "Eye movement event type.",
                        "Levels": {
                            "fixation": "Fixation detected by the EyeLink event parser.",
                            "blink": "Blink detected by the EyeLink event parser.",
                        },
                    },
                    "blink": {
                        "Description": "One for blink events and zero for fixation events."
                    },
                    "x_coordinate": {
                        "Description": "Mean fixation x-coordinate; n/a for blinks.",
                        "Units": "pixel",
                    },
                    "y_coordinate": {
                        "Description": "Mean fixation y-coordinate; n/a for blinks.",
                        "Units": "pixel",
                    },
                    "image_name": {"Description": "Stimulus filename."},
                    "image_modality": {"Description": "Stimulus modality."},
                    "trial_key": {"Description": "Project-specific trial identifier."},
                    "event_id": {
                        "Description": "Fixation or blink index within the trial."
                    },
                },
                beh_dir / f"{events_stem}.json",
            )

    derivative_dir = (
        derivative_root / f"sub-{participant}" / "beh"
    )
    derivative_dir.mkdir(parents=True, exist_ok=True)

    derivative_columns = [
        column for column in [
            "participant_id",
            "trial_key",
            "trial_index",
            "image_modality",
            "image_id",
            "image_stem",
            "image_name",
            "fixation_id",
            "eye",
            "relative_start",
            "relative_end",
            "duration",
            "x",
            "y",
            "x_stimulus",
            "y_stimulus",
            "is_inside_screen",
            "is_inside_stimulus",
            "screen_width",
            "screen_height",
            "stimuli_width",
            "stimuli_height",
            "image_width",
            "image_height",
        ]
        if column in clean_fixations.columns
    ]
    write_tsv(
        clean_fixations[derivative_columns],
        derivative_dir / f"{prefix}_desc-clean_fixations.tsv",
        header=True,
    )

    participant_rows.append({
        "participant_id": f"sub-{participant}",
        "modality": modality,
    })
    build_rows.append({
        "participant_id": participant,
        "modality": modality,
        "run": run,
        "asc_file": asc_path.name,
        "n_trials": len(raw_summary),
        "n_raw_fixations": len(raw_fixations),
        "n_clean_fixations": len(clean_fixations),
        "sampling_frequency": sampling_frequency,
        "recorded_eyes": ",".join(available_eyes),
    })

participants = pd.DataFrame(participant_rows)
participants = (
    participants.groupby("participant_id", as_index=False)
    .agg(modality=("modality", lambda values: "+".join(sorted(set(values)))))
)

if not source_participants.empty and "participant_id" in source_participants.columns:
    extra_columns = [
        column for column in source_participants.columns
        if column not in {"id", "participant_id"}
    ]
    participants = participants.merge(
        source_participants[["participant_id", *extra_columns]].rename(
            columns={"participant_id": "_participant_label"}
        ),
        left_on=participants["participant_id"].str.removeprefix("sub-"),
        right_on="_participant_label",
        how="left",
    )
    participants = participants.drop(
        columns=["key_0", "_participant_label"],
        errors="ignore",
    )

write_tsv(participants, BIDS_DIR / "participants.tsv", header=True)
write_json(
    {
        "participant_id": {
            "Description": "Unique anonymized BIDS participant identifier."
        },
        "modality": {
            "Description": "Stimulus modality viewed by the participant."
        },
        "subset": {
            "Description": "Stimulus subset assigned during acquisition."
        },
    },
    BIDS_DIR / "participants.json",
)

(BIDS_DIR / "README").write_text(
    "DeepGazeIR eye-tracking dataset formatted according to BIDS 1.11.1.\n"
    "Raw continuous gaze samples and device-derived events are stored under "
    "sub-*/beh/. Cleaned fixation tables are stored under derivatives/.\n",
    encoding="utf-8",
)

build_summary = pd.DataFrame(build_rows)
build_summary

Profiling trials: 100%|██████████| 150/150 [00:00<00:00, 461.76it/s]


,participant_id,modality,run,asc_file,n_trials,n_raw_fixations,n_clean_fixations,sampling_frequency,recorded_eyes
0,001,infrared,1,ID-001-2026_04_20_09_27.asc,250,2087,1556,20.618404,right
1,002,infrared,1,ID-002-2026_04_20_10_09.asc,250,2541,1505,333.333333,right
2,003,visible,1,ID-003-2026_04_20_11_34.asc,150,1381,734,24.096176,right
3,004,infrared,1,ID-004-2026_04_20_10_55.asc,250,2228,1869,500.000000,right
4,005,infrared,1,ID-005-2026_04_20_12_22.asc,250,2378,2114,28.985205,right
...,...,...,...,...,...,...,...,...,...
63,068,infrared,1,ID-068-2026_06_17_14_47.asc,250,2978,2517,200.000000,left
64,069,visible,1,ID-069-2026_06_18_10_30.asc,150,1625,1428,16.948946,left
65,070,infrared,1,ID-070-2026_06_18_14_05.asc,250,2424,2146,33.332933,left
66,071,infrared,1,ID-071-2026_06_18_14_42.asc,250,2084,1768,37.735336,left


In [13]:
required_root_files = [
    BIDS_DIR / "dataset_description.json",
    BIDS_DIR / "participants.tsv",
    BIDS_DIR / "participants.json",
]

missing = [path for path in required_root_files if not path.exists()]
if missing:
    raise RuntimeError(f"Missing required files: {missing}")

physio_files = sorted(BIDS_DIR.glob("sub-*/beh/*_physio.tsv.gz"))
events_files = sorted(BIDS_DIR.glob("sub-*/beh/*_events.tsv"))
physioevents_files = sorted(BIDS_DIR.glob("sub-*/beh/*_physioevents.tsv.gz"))

if not physio_files:
    raise RuntimeError("No BIDS physio file was created.")
if not events_files:
    raise RuntimeError("No BIDS task events file was created.")

print(f"Participants:       {participants['participant_id'].nunique()}")
print(f"Runs:               {len(build_summary)}")
print(f"Physio files:       {len(physio_files)}")
print(f"Task events files:  {len(events_files)}")
print(f"Physioevents files: {len(physioevents_files)}")
print(f"Output directory:   {BIDS_DIR}")

print("\nTop-level tree:")
for path in sorted(BIDS_DIR.rglob("*")):
    relative = path.relative_to(BIDS_DIR)
    if len(relative.parts) <= 3:
        suffix = "/" if path.is_dir() else ""
        print(f"  {relative}{suffix}")

print(
    "\nExternal validation command:\n"
    f"npx bids-validator \"{BIDS_DIR}\""
)

Participants:       68
Runs:               68
Physio files:       68
Task events files:  68
Physioevents files: 68
Output directory:   /Users/noe-edouard/Desktop/ETUDES/INP/Stages/3A/Code/data/2026/bids

Top-level tree:
  .DS_Store
  README
  dataset_description.json
  derivatives/
  derivatives/deepgazeir-processing/
  derivatives/deepgazeir-processing/dataset_description.json
  derivatives/deepgazeir-processing/sub-001/
  derivatives/deepgazeir-processing/sub-002/
  derivatives/deepgazeir-processing/sub-003/
  derivatives/deepgazeir-processing/sub-004/
  derivatives/deepgazeir-processing/sub-005/
  derivatives/deepgazeir-processing/sub-007/
  derivatives/deepgazeir-processing/sub-008/
  derivatives/deepgazeir-processing/sub-009/
  derivatives/deepgazeir-processing/sub-010/
  derivatives/deepgazeir-processing/sub-011/
  derivatives/deepgazeir-processing/sub-012/
  derivatives/deepgazeir-processing/sub-013/
  derivatives/deepgazeir-processing/sub-014/
  derivatives/deepgazeir-processin

## Notes

- `*_physio.tsv.gz` contains continuous gaze samples and has no header, as required by BIDS.
- `*_physioevents.tsv.gz` contains EyeLink fixation and blink events.
- `*_events.tsv` contains stimulus presentation events.
- Cleaned fixation data are derivatives and are therefore kept outside the raw BIDS files.
- Verify the physical screen dimensions, viewing distance, refresh rate, tracker model, and sampling frequency before publication.
- Run the official BIDS Validator after dataset construction.
